# OdiaEval — Group 9 — Natural Language Inference

Fine-tunes `ai4bharat/IndicBERTv2-MLM-only` on the Odia split of **IndicXNLI**, reproduces the published **72.6%** accuracy, then evaluates the same frozen model on the native Odia test set to measure the translationese gap.

**Before you start:** Runtime → Change runtime type → **T4 GPU** (or better).

Expected training time: ~3.5–5 h on a T4, ~1.5 h on an A100. The notebook saves checkpoints every epoch, so a disconnect is recoverable — rerun the training cell with `--resume`.


## 1. Setup

In [ ]:
!nvidia-smi
import torch
print("CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU."


In [ ]:
# Clone the repo. Replace with your own fork once you have pushed it.
REPO_URL = "https://github.com/<your-username>/odiaeval-group9-nli.git"

import os
if not os.path.exists("odiaeval-group9-nli"):
    !git clone $REPO_URL
%cd odiaeval-group9-nli
!pip install -q -r requirements.txt


## 2. Data

The IndicXNLI Odia parquet files are not in the repo (too large for git). Get them either from Google Drive, or straight from the HuggingFace dataset.

Run **one** of the two cells below.

In [ ]:
# Option A - from your Google Drive (upload the three parquet files there first)
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p data/benchmark
!cp "/content/drive/MyDrive/odiaeval/indicxnli-train.parquet"      data/benchmark/
!cp "/content/drive/MyDrive/odiaeval/indicxnli-validation.parquet" data/benchmark/
!cp "/content/drive/MyDrive/odiaeval/indicxnli-test.parquet"       data/benchmark/
!ls -la data/benchmark/


In [ ]:
# Option B - download the Odia split directly from HuggingFace
from datasets import load_dataset
import os

os.makedirs("data/benchmark", exist_ok=True)
ds = load_dataset("Divyanshu/indicxnli", "or")   # 'or' = Odia
for split, name in [("train","train"), ("validation","validation"), ("test","test")]:
    df = ds[split].to_pandas()[["premise","hypothesis","label"]]
    df.to_parquet(f"data/benchmark/indicxnli-{name}.parquet", index=False)
    print(f"{name}: {len(df):,} rows")


## 3. Integrity gate

Checks that the splits are the right size, the text really is Odia, and no training pair leaks into an evaluation split. Writes `results/dataset_stats.json`.

Training refuses to start if any of these fail.

In [ ]:
!python scripts/verify_data.py

## 4. Fine-tune

Train on the 392,702-pair training split only. Checkpoint selection uses the 2,490-pair validation split. **The test split is not loaded by this script at all.**

Hyperparameters are pinned in `configs/indicbertv2_odia.yaml`, taken from Doddapaneni et al. (ACL 2023), Table 13.

In [ ]:
!python -m src.train

In [ ]:
# If Colab disconnected mid-run, rerun with --resume instead.
# !python -m src.train --resume


### Back up the trained model

Colab wipes local storage when the session ends. Copy the model to Drive now — the native evaluation needs it, and the brief requires a saved model.

In [ ]:
!mkdir -p "/content/drive/MyDrive/odiaeval/artifacts"
!cp -r artifacts/indicbertv2-odia-nli "/content/drive/MyDrive/odiaeval/artifacts/"
!du -sh "/content/drive/MyDrive/odiaeval/artifacts/indicbertv2-odia-nli"


## 5. Benchmark reproduction

Evaluate once on the held-out 5,010-pair test set and compare against the published 72.6%.

Target: within ±2 points, i.e. **70.6% – 74.6%**.

In [ ]:
!python -m src.evaluate --split benchmark_test

## 6. Native Odia evaluation

Same saved model, same preprocessing, same label mapping, same metric. Only the data changes.

Upload the professor's native Odia test set into `data/native/` first. `src/data.py` accepts csv, tsv, json, jsonl, parquet and xlsx, and handles the common column spellings (`sentence1`/`sentence2`/`gold_label` as well as `premise`/`hypothesis`/`label`).

In [ ]:
from google.colab import files
import os
os.makedirs("data/native", exist_ok=True)
uploaded = files.upload()
for name in uploaded:
    os.rename(name, f"data/native/{name}")
    print("saved to data/native/" + name)


In [ ]:
NATIVE_FILE = "data/native/<filename>"   # <- set this
!python -m src.evaluate --split native --native-file $NATIVE_FILE


## 7. Translationese gap and final report

Writes `results/final_report.json` and `docs/RESULTS.md` with every table the brief asks for.

In [ ]:
!python -m src.report

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("docs/RESULTS.md").read()))


## 8. Commit the evidence

`results/` is the proof behind the claimed numbers. Push it — two other groups were sent back this term for missing exactly these files.

In [ ]:
!git config --global user.email "you@example.com"
!git config --global user.name  "Your Name"

!git add results/ docs/RESULTS.md
!git commit -m "Add Group 9 results: benchmark + native Odia evaluation"
# Needs a GitHub personal access token as the password:
!git push


### Also attach the trained model

Model weights are too large for git. Upload `artifacts/indicbertv2-odia-nli/` as a **GitHub Release asset**, or share a Drive link, and record the link in `docs/ARTIFACTS.md`.